# 15 — Price co-movement

Estimate Portugal-Spain retail price co-movement using weekly pre-tax prices. The notebook requires a tidy price-history extraction and persists coefficients and diagnostics.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from portugal_refining_resilience.config import get_paths, load_analysis_config
from portugal_refining_resilience.io import persist_dataframe, write_json

PATHS = get_paths(ROOT)
pd.set_option("display.max_columns", 100)

from portugal_refining_resilience.prices import adf_lag_rule_sensitivity, choose_price_model, model_choice_scale_comparison, fit_error_correction_model, fit_price_comovement, fit_short_run_price_transmission, price_comovement_design, spread_stationarity, stationarity_diagnostics, kpss_levels_diagnostics, cointegrating_slope_test, elasticity_unit_tests, post_period_adjustment_stability, spread_stationarity_by_regime, weekly_coverage, gregory_hansen_test, second_break_test, cross_country_placebo, false_break_placebo, extract_weekly_prices, PLACEBO_COUNTRIES


## Required tidy input contract

Create `data/interim/weekly_oil_prices_tidy.csv` from the Commission workbook with:

`date, country, product, price_with_tax_eur_per_1000l, price_without_tax_eur_per_1000l`

for at least `PT` and `ES`, and products `diesel` and `gasoline`.

The source workbook is deliberately inventoried in notebook 05 because historical sheet layouts can change. Never silently guess columns if the workbook changes.


In [ ]:
path = PATHS.interim / "weekly_oil_prices_tidy.csv"
if not path.exists():
    raise FileNotFoundError(
        "Missing data/interim/weekly_oil_prices_tidy.csv. Extract it from the audited EC workbook before running price models."
    )
prices = pd.read_csv(path, parse_dates=["date"])
# The Commission keeps publishing after the study window closes. Without this the
# price arm silently runs past end_year while the paper states 2005 to 2024, and
# the post-transition period grows by every week the bulletin adds.
_price_config = load_analysis_config(ROOT)
prices = prices.loc[
    prices["date"].dt.year.between(
        int(_price_config["price_start_year"]), int(_price_config["end_year"])
    )
]
required = {"date", "country", "product", "price_without_tax_eur_per_1000l"}
if not required.issubset(prices.columns):
    raise ValueError(f"Price input missing: {sorted(required - set(prices.columns))}")


In [ ]:
stationarity = stationarity_diagnostics(
    prices,
    value_column="price_without_tax_eur_per_1000l",
    group_columns=["country", "product"],
)
persist_dataframe(stationarity, PATHS.metrics / "price_stationarity_diagnostics.csv")
display(stationarity)


In [ ]:
# PT pre-tax price on Spain pre-tax price, with a post-transition interaction for changed co-movement.
rows = []
choice_rows = []
lag_rule_rows = []
scale_rows = []
spread_rows = []
short_run_rows = []
ecm_rows = []
half_life_rows = []
for product in ["diesel", "gasoline"]:
    wide = price_comovement_design(prices, product=product)
    choice = choose_price_model(wide, product=product)
    choice_rows.append(choice)
    # The gasoline verdict is marginal, so record it under every lag rule a
    # reader might reasonably have chosen rather than only the one used.
    lag_rule_rows.append(adf_lag_rule_sensitivity(wide, product=product))
    scale_rows.append(model_choice_scale_comparison(wide, product=product))
    # The levels regression is persisted either way, but never without the verdict
    # that says whether levels are admissible. Anything other than "levels" means
    # the short-run log-difference model below carries the inference.
    family = str(choice["model_family"])
    levels_valid = family == "levels"
    spread_rows.append({"product": product, **spread_stationarity(wide)})
    model = fit_price_comovement(wide)
    for term in model.params.index:
        rows.append({"product": product, "term": term, "estimate": model.params[term], "std_error": model.bse[term], "p_value": model.pvalues[term], "nobs": model.nobs, "covariance": "HAC(8)", "outcome": "PT pre-tax EUR/1000L", "comparison": "ES pre-tax EUR/1000L", "model": "PT-ES price co-movement with ES_x_post interaction", "model_family": family, "levels_model_valid": levels_valid})
    # An ECM is only fitted where the diagnostics licence one. Where cointegration
    # is rejected the disequilibrium term is not a valid long-run residual, so
    # fitting it anyway would contradict the model choice recorded above.
    if family == "ecm_required":
        ecm = fit_error_correction_model(wide)
        ecm_model = ecm["model"]
        for term in ecm_model.params.index:
            ecm_rows.append({
                "product": product, "term": term,
                "estimate": ecm_model.params[term], "std_error": ecm_model.bse[term],
                "p_value": ecm_model.pvalues[term], "nobs": ecm_model.nobs,
                "cointegrating_constant": ecm["cointegrating_constant"],
                "cointegrating_slope": ecm["cointegrating_slope"],
                "covariance": "HAC(8)", "model_family": family,
                "model": "two-step Engle-Granger error-correction model",
            })
        # The paper quotes post-transition levels and half-lives, which are sums and
        # transforms of the fitted terms. Persist them with Wald tests so every figure
        # in the price section exists in the evidence rather than being arithmetic the
        # reader has to redo.
        for term, label in (
            ("disequilibrium_lag", "disequilibrium_lag_post_period"),
            ("diff_log_ES", "diff_log_ES_post_period"),
        ):
            wald = ecm_model.t_test(f"{term} + {term}_x_post = 0")
            ecm_rows.append({
                "product": product, "term": label,
                "estimate": float(wald.effect[0]), "std_error": float(wald.sd[0][0]),
                "p_value": float(wald.pvalue), "nobs": ecm_model.nobs,
                "cointegrating_constant": ecm["cointegrating_constant"],
                "cointegrating_slope": ecm["cointegrating_slope"],
                "covariance": "HAC(8)", "model_family": family,
                "model": "two-step Engle-Granger error-correction model, post-period level (Wald)",
            })
        speed_pre = float(ecm_model.params["disequilibrium_lag"])
        speed_post = speed_pre + float(ecm_model.params["disequilibrium_lag_x_post"])
        for phase, speed in (("pre_transition", speed_pre), ("post_transition", speed_post)):
            half_life_rows.append({
                "product": product, "phase": phase,
                "adjustment_speed": speed,
                "half_life_weeks": float(np.log(2) / -np.log(1 + speed)),
                "model": "two-step Engle-Granger error-correction model",
            })
    short_model = fit_short_run_price_transmission(wide)
    for term in short_model.params.index:
        short_run_rows.append({"product": product, "term": term, "estimate": short_model.params[term], "std_error": short_model.bse[term], "p_value": short_model.pvalues[term], "nobs": short_model.nobs, "covariance": "HAC(8)", "outcome": "delta log PT pre-tax price", "comparison": "delta log ES pre-tax price", "model": "short-run log-difference transmission", "model_family": family})
    # The paper quotes the post-transition elasticity, which is the pre-period
    # slope plus the interaction. Persist it as its own row with a Wald test so
    # the figure exists in the evidence rather than being a sum readers must do.
    post_level = short_model.t_test("diff_log_ES + diff_log_ES_x_post = 0")
    short_run_rows.append({"product": product, "term": "diff_log_ES_post_period", "estimate": float(post_level.effect[0]), "std_error": float(post_level.sd[0][0]), "p_value": float(post_level.pvalue), "nobs": short_model.nobs, "covariance": "HAC(8)", "outcome": "delta log PT pre-tax price", "comparison": "delta log ES pre-tax price", "model": "short-run log-difference transmission, post-period level (Wald)", "model_family": family})
coefs = pd.DataFrame(rows)
persist_dataframe(coefs, PATHS.metrics / "price_comovement_models.csv")
choices = pd.DataFrame(choice_rows)
persist_dataframe(choices, PATHS.metrics / "price_model_choice.csv", key_columns=["product"])
lag_sensitivity = pd.concat(lag_rule_rows, ignore_index=True)
persist_dataframe(lag_sensitivity, PATHS.metrics / "price_adf_lag_sensitivity.csv", key_columns=["product", "country", "regression", "lag_rule"])
display(lag_sensitivity)
scale_comparison = pd.concat(scale_rows, ignore_index=True)
persist_dataframe(scale_comparison, PATHS.metrics / "price_model_choice_scale_comparison.csv", key_columns=["product", "scale"])
display(scale_comparison)
spread_tests = pd.DataFrame(spread_rows)
persist_dataframe(spread_tests, PATHS.metrics / "pt_es_spread_stationarity.csv", key_columns=["product", "diagnostic"])
ecm_models = pd.DataFrame(ecm_rows)
if not ecm_models.empty:
    persist_dataframe(ecm_models, PATHS.metrics / "price_ecm_models.csv", key_columns=["product", "term"])
    display(ecm_models[["product", "term", "estimate", "std_error", "p_value"]])
half_lives = pd.DataFrame(half_life_rows)
if not half_lives.empty:
    persist_dataframe(half_lives, PATHS.metrics / "price_ecm_half_lives.csv", key_columns=["product", "phase"])
    display(half_lives)
short_run = pd.DataFrame(short_run_rows)
persist_dataframe(short_run, PATHS.metrics / "price_short_run_models.csv")
display(coefs)


In [ ]:
# Four diagnostics the price arm was previously asked to do without. Each answers a
# question a reader can otherwise only raise as an objection: whether the marginal
# gasoline ADF verdict survives reversing the null, whether the long-run slope is
# distinguishable from one (which decides how the EUR spread may be read), whether the
# elasticity crosses one rather than merely rising, and whether the post-transition
# adjustment speed is a property of the regime or of the 2022 episode inside it.
kpss_rows = []
slope_rows = []
elasticity_rows = []
stability_rows = []
regime_spread_rows = []
for product in ["diesel", "gasoline"]:
    wide = price_comovement_design(prices, product=product)
    kpss_rows.append(kpss_levels_diagnostics(wide, product=product))
    regime_spread_rows.append(spread_stationarity_by_regime(wide, product=product))
    if str(choose_price_model(wide, product=product)["model_family"]) == "ecm_required":
        slope_rows.append(cointegrating_slope_test(wide, product=product))
        elasticity_rows.append(elasticity_unit_tests(wide, product=product))
        stability_rows.append(post_period_adjustment_stability(wide, product=product))

kpss_tests = pd.concat(kpss_rows, ignore_index=True)
persist_dataframe(kpss_tests, PATHS.metrics / "price_kpss_diagnostics.csv", key_columns=["product", "country", "regression"])
display(kpss_tests)

regime_spreads = pd.concat(regime_spread_rows, ignore_index=True)
persist_dataframe(regime_spreads, PATHS.metrics / "pt_es_spread_stationarity_by_regime.csv", key_columns=["product", "segment"])
display(regime_spreads)

if slope_rows:
    slope_tests = pd.concat(slope_rows, ignore_index=True)
    persist_dataframe(slope_tests, PATHS.metrics / "price_cointegrating_slope_tests.csv", key_columns=["product", "leads_lags"])
    display(slope_tests)
if elasticity_rows:
    elasticity_tests = pd.concat(elasticity_rows, ignore_index=True)
    persist_dataframe(elasticity_tests, PATHS.metrics / "price_elasticity_unit_tests.csv", key_columns=["product", "phase"])
    display(elasticity_tests)
if stability_rows:
    stability = pd.concat(stability_rows, ignore_index=True)
    persist_dataframe(stability, PATHS.metrics / "price_post_period_stability.csv", key_columns=["product", "subset"])
    display(stability)


In [ ]:
# The models read the weekly index as evenly spaced and the bulletin does not supply it
# that way, so the size of the gap has to be on the record rather than in a sentence.
coverage = weekly_coverage(prices)
persist_dataframe(coverage, PATHS.metrics / "weekly_price_coverage.csv", key_columns=["country", "product"])
display(coverage)


In [ ]:
# Two assumptions the price arm makes and had never tested. The first is that one
# cointegrating vector holds across a date the rest of the model treats as a break;
# the second is that the price relation breaks once, at the closure, while the
# physical arm partitions the same period into four phases.
shift_rows = []
second_rows = []
for product in ["diesel", "gasoline"]:
    wide = price_comovement_design(prices, product=product)
    if str(choose_price_model(wide, product=product)["model_family"]) != "ecm_required":
        continue
    shift_rows.append(gregory_hansen_test(wide, product=product))
    second_rows.append(second_break_test(wide, product=product))

if shift_rows:
    regime_shift = pd.concat(shift_rows, ignore_index=True)
    persist_dataframe(regime_shift, PATHS.metrics / "price_regime_shift_cointegration.csv", key_columns=["product"])
    display(regime_shift)
if second_rows:
    second_break = pd.concat(second_rows, ignore_index=True)
    persist_dataframe(second_break, PATHS.metrics / "price_second_break_tests.csv", key_columns=["product", "term"])
    display(second_break)


In [ ]:
# The adjustment speed quadruples after May 2021. Whether that is Portuguese or is
# just what happened to every European pair after 2021 is answerable, because the
# bulletin prices every member state. Neighbours that closed no refinery are the
# comparison, and false breaks on pre-closure data are the other half of the test.
_bulletin = PATHS.raw / "ec_weekly_oil_bulletin" / "weekly_oil_bulletin_price_history.xlsx"
if _bulletin.exists():
    _wide_panel = extract_weekly_prices(_bulletin, countries=("PT", "ES", *PLACEBO_COUNTRIES))
    _wide_panel = _wide_panel.loc[
        _wide_panel["date"].dt.year.between(
            int(_price_config["price_start_year"]), int(_price_config["end_year"])
        )
    ]
    persist_dataframe(
        _wide_panel,
        PATHS.interim / "weekly_oil_prices_placebo_panel.csv",
        key_columns=["date", "country", "product"],
        metadata={
            "unit": "EUR per 1000 litres",
            "role": "placebo comparison only; the reported models use PT and ES",
        },
    )
    cross_country = cross_country_placebo(_wide_panel)
    persist_dataframe(cross_country, PATHS.metrics / "price_cross_country_placebo.csv", key_columns=["pair", "product"])
    display(cross_country)

    false_break = false_break_placebo(_wide_panel)
    persist_dataframe(false_break, PATHS.metrics / "price_false_break_placebo.csv", key_columns=["break_date", "product"])
    display(false_break)
else:
    print("Bulletin workbook absent; placebo tests skipped.")
